In [1]:
import os
from datetime import datetime

input_file = "server_logs.csv"
report_file = "summary_report.txt"
audit_file = "audit.log"

LATENCY_THRESHOLD = 500

os.makedirs("reports", exist_ok=True)
os.makedirs("logs", exist_ok=True)

valid_records = 0
success = 0
errors = 0
total_latency = 0

servers = {}

try:
    with open(input_file, "r") as file:

        file.readline()

        line_number = 2

        for line in file:

            line = line.strip()
            parts = line.split(",")

            try:
                timestamp = parts[0]
                server_id = parts[1]

                response_time = float(parts[2])
                status_code = int(parts[3])

            except:
                print("Warning [Line", line_number, "]: Malformed value skipped.")
                line_number += 1
                continue

            valid_records += 1
            total_latency += response_time

            if status_code == 200:
                success += 1

            if status_code >= 400:
                errors += 1

            if server_id not in servers:
                servers[server_id] = [0, 0]

            servers[server_id][0] += response_time
            servers[server_id][1] += 1

            line_number += 1

except FileNotFoundError:
    print("Error: server_logs.csv not found.")
    exit()

if valid_records > 0:
    average_latency = total_latency / valid_records
else:
    average_latency = 0

with open(report_file, "w") as report:

    report.write("=== CLOUD SERVER LOG ANALYSIS REPORT ===\n")
    report.write("Valid Records Processed: " + str(valid_records) + "\n")
    report.write("Successful Requests (200 OK): " + str(success) + "\n")
    report.write("Error Responses (4xx/5xx): " + str(errors) + "\n")
    report.write("Overall Average Latency: {:.2f} ms\n\n".format(average_latency))

    report.write("--- Per-Server Latency Breakdown ---\n")

    for server in servers:

        total = servers[server][0]
        count = servers[server][1]

        avg = total / count

        report.write(
            "Server " + server +
            ": " + str(count) +
            " requests, Avg Latency: {:.2f} ms\n".format(avg)
        )

    report.write("\n--- Latency Alerts ---\n")

    for server in servers:

        avg = servers[server][0] / servers[server][1]

        if avg > LATENCY_THRESHOLD:
            report.write(server + " exceeds latency threshold\n")

if os.path.exists(audit_file):

    if os.path.getsize(audit_file) > 2048:

        time_stamp = datetime.now().strftime("%Y%m%d_%H%M%S")

        new_name = "logs/audit_archive_" + time_stamp + ".log"

        os.rename(audit_file, new_name)

with open(audit_file, "a") as audit:

    current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    audit.write(
        "[" + current_time + "] SUCCESS: Processed "
        + str(valid_records)
        + " logs. Avg: "
        + "{:.2f}".format(average_latency)
        + "ms.\n"
    )

print("Processing complete.")

Error: server_logs.csv not found.
Processing complete.
